# Introduccion al Analisis Molecular — RuBisCO

Este notebook demuestra el uso basico de la libreria para:
- Obtener PDBs de RuBisCO (familias G-I a G-IV)
- Visualizar estructuras con py3Dmol
- Analisis basico de propiedades estructurales

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
import urllib.request
from pathlib import Path
from Bio.PDB import PDBParser

from libreria_analisismolecular import utils, colab

print('Imports OK')

## 1. PDBs de RuBisCO por familia

In [ ]:
# RuBisCO Form I — familias y especificidad CO2/O2 (S)
# Fuente: Poudel et al. (2020), Tabla S1
RUBISCO_PDBS = {
    '4RUB':  {'group': 'G-I (cyanobacteria)', 'S': 77, 'organism': 'Synechococcus'},
    '1GK8':  {'group': 'G-I (cyanobacteria)', 'S': 61, 'organism': 'Synechocystis'},
    '1RBL':  {'group': 'G-II (proteobacteria)', 'S': 13, 'organism': 'Rhodospirillum'},
    '1GEH':  {'group': 'G-II (proteobacteria)', 'S': 15, 'organism': 'Rhodospirillum'},
    '2RUS':  {'group': 'G-III (archaea)',     'S': 5,  'organism': 'Thermococcus'},
    '8RUB':  {'group': 'G-I (plant)',         'S': 86, 'organism': 'Spinacia (spinach)'},
}

df_pdb = pd.DataFrame.from_dict(RUBISCO_PDBS, orient='index')
df_pdb.index.name = 'PDB'
display(df_pdb)

# Descargar PDBs
PDB_DIR = Path('/content/pdb')
PDB_DIR.mkdir(exist_ok=True)

for pid in RUBISCO_PDBS:
    fp = PDB_DIR / f'{pid}.pdb'
    if not fp.exists():
        url = f'https://files.rcsb.org/download/{pid}.pdb'
        urllib.request.urlretrieve(url, fp)
        print(f'Descargado {pid}')

print(f'\nTotal PDBs: {len(list(PDB_DIR.glob("*.pdb")))}')

## 2. Visualizar estructura 3D con py3Dmol

In [ ]:
pdb_to_view = '4RUB'  # Cambiar para ver otro PDB
pdb_content = open(PDB_DIR / f'{pdb_to_view}.pdb').read()

view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_content, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

info = RUBISCO_PDBS[pdb_to_view]
print(f'{pdb_to_view} — {info["organism"]} (Grupo: {info["group"]}, S={info["S"]})')

## 3. Analisis basico de estructura

In [ ]:
parser = PDBParser(QUIET=True)

structures = []
for pid in RUBISCO_PDBS:
    pdb_path = PDB_DIR / f'{pid}.pdb'
    if not pdb_path.exists():
        continue
    
    try:
        struct = parser.get_structure(pid, str(pdb_path))
        model = struct[0]
        
        n_chains = len(model)
        n_residues = sum(1 for _ in model.get_residues())
        n_atoms = sum(1 for _ in model.get_atoms())
        
        structures.append({
            'pdb': pid,
            'chains': n_chains,
            'residues': n_residues,
            'atoms': n_atoms,
        })
    except Exception as e:
        print(f'Error con {pid}: {e}')

df_struct = pd.DataFrame(structures)
if not df_struct.empty:
    display(df_struct)

## 4. Especificidad CO2/O2 por grupo

In [ ]:
plt.figure(figsize=(10, 5))
df_pdb_sorted = df_pdb.sort_values('S', ascending=True)
colors = ['#e74c3c' if 'G-III' in str(g) else '#f39c12' if 'G-II' in str(g) else '#2ecc71' for g in df_pdb_sorted['group']]

plt.barh(df_pdb_sorted.index, df_pdb_sorted['S'], color=colors, edgecolor='black')
for i, (idx, row) in enumerate(df_pdb_sorted.iterrows()):
    plt.text(row['S'] + 1, i, f'S={row["S"]}', va='center', fontweight='bold')

plt.xlabel('Especificidad CO2/O2 (S)')
plt.title('Especificidad de RuBisCO por familia')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Observaciones:')
print('  G-I (plant/cyano): mayor S, mejor discriminacion CO2/O2')
print('  G-II (proteobacteria): S intermedia')
print('  G-III (archaea): menor S, casi sin discriminacion')

## 5. Guardar resultados en Drive (opcional)

In [ ]:
# Guardar tabla de PDBs
df_pdb.to_csv('/content/rubisco_pdbs.csv')
df_struct.to_csv('/content/rubisco_structures.csv')

# Descomentar para guardar en Drive:
# colab.drive_utils.save_to_drive('/content/rubisco_pdbs.csv', 'rubisco')
# colab.drive_utils.save_to_drive('/content/rubisco_structures.csv', 'rubisco')

print('Datos guardados localmente')
print('\nProximo paso: Flujo completo VS Code → Colab → Drive (02_workflow.ipynb)')